# 2.3 Gene Expression Analysis & 2.4 Functional Analysis

### Prerequisites & Setup
Ensure you have `fastqc`, `salmon`, `cufflinks` (Tuxedo), R (with DESeq2), and Python libraries installed. 

```python
# Install required Python libraries
!pip install pandas numpy scipy matplotlib seaborn rpy2 requests gseapy
```

In [4]:
import pandas as pd
import numpy as np

# 1. Load the Tuxedo (Cuffdiff) results directly
# Pandas can read .gz files natively if you haven't extracted it
diff_file = "GSE122125_gene_exp.diff.gz" 
df = pd.read_csv(diff_file, sep='\t')

# 2. Keep only successful statistical tests
df = df[df['status'] == 'OK']

# 3. Apply significance thresholds: FDR (q_value in Cuffdiff) < 0.05 and |log2FoldChange| > 2
# Note: Cuffdiff column names are 'q_value' and 'log2(fold_change)'
degs = df[(df['q_value'] < 0.05) & (df['log2(fold_change)'].abs() > 2)]
print(f"Total significant DEGs before biotype filtering: {len(degs)}")

# 4. Filter out unwanted RNA types (rRNA, miRNA, piRNA, etc.)
# If you have a WormBase biotype mapping file, merge it here.
# For demonstration, we assume a 'gene_biotypes.csv' mapping Ensembl IDs to biotypes
try:
    biotypes = pd.read_csv("gene_biotypes.csv") 
    degs_merged = degs.merge(biotypes, left_on='gene_id', right_on='gene_id', how='left')
    unwanted_types = ['miRNA', 'piRNA', 'rRNA', 'ncRNA', 'unconfirmed']
    final_degs = degs_merged[~degs_merged['biotype'].isin(unwanted_types)]
except FileNotFoundError:
    print("Biotype file not found. Proceeding with all DEGs.")
    final_degs = degs

print(f"Final DEGs for functional analysis: {len(final_degs)}")

# 5. Export for Functional Analysis (DAVID, KEGG, Cytoscape)
# Extract just the gene identifiers (usually 'gene' or 'gene_id' column)
gene_list = final_degs['gene_id'].dropna().unique().tolist()

with open('deg_gene_list.txt', 'w') as f:
    for gene in gene_list:
        f.write(f"{gene}\n")
        
print("Gene list exported to 'deg_gene_list.txt' for functional analysis tools.")

Total significant DEGs before biotype filtering: 10796
Biotype file not found. Proceeding with all DEGs.
Final DEGs for functional analysis: 10796
Gene list exported to 'deg_gene_list.txt' for functional analysis tools.


Filtering and Clustering (Python)
Filter out unconfirmed genes and ncRNAs, apply threshold (FDR < 0.05, log2FC > 2), and generate a dendrogram using Ward's method.

In [8]:
import pandas as pd
import numpy as np

# 1. Load the Tuxedo (Cuffdiff) results directly from the DIFF file
diff_file = "GSE122125_gene_exp.diff.gz" # Pandas can read .gz directly
df = pd.read_csv(diff_file, sep='\t')

# 2. Keep only successful statistical tests from the pipeline
df = df[df['status'] == 'OK']

# 3. Filter out rRNA, miRNA, piRNA 
# Assuming you have a mapping file 'gene_biotypes.csv' mapping 'gene_id' to 'biotype'
try:
    biotypes = pd.read_csv("gene_biotypes.csv") 
    df_merged = df.merge(biotypes, on='gene_id', how='left')
    unwanted_types = ['miRNA', 'piRNA', 'rRNA', 'ncRNA', 'unconfirmed']
    df_filtered = df_merged[~df_merged['biotype'].isin(unwanted_types)]
except FileNotFoundError:
    print("Biotype file not found. Proceeding without biotype filtering.")
    df_filtered = df

# 4. Apply significance thresholds: FDR (q_value) < 0.05 and |log2FoldChange| > 2
# Note: Cuffdiff uses 'q_value' for FDR and 'log2(fold_change)' for log2FC
degs = df_filtered[(df_filtered['q_value'] < 0.05) & (df_filtered['log2(fold_change)'].abs() > 2)]
print(f"Number of DEGs identified: {len(degs)}")

# 5. Save final DEG list for functional analysis
degs.to_csv("filtered_DEGs.csv", index=False)

# 6. Export just the gene list for web tools (DAVID, KEGG, Cytoscape)
gene_list = degs['gene_id'].dropna().unique().tolist()
with open('deg_gene_list.txt', 'w') as f:
    for gene in gene_list:
        f.write(f"{gene}\n")
print("Gene list exported to 'deg_gene_list.txt' for functional analysis.")

Biotype file not found. Proceeding without biotype filtering.
Number of DEGs identified: 10796
Gene list exported to 'deg_gene_list.txt' for functional analysis.


Functional Analysis Preparation & Execution (Python)
The tools requested (DAVID, Cytoscape/BINGO, KEGG Mapper, WormBase GSEA) are web-based or standalone GUI applications. Below is the Python code to extract the gene identifiers for these platforms, and an automated equivalent using the gseapy library.

In [11]:
import gseapy as gp
from IPython.display import display

# Extract WormBase gene IDs from the specific Cuffdiff column
gene_list = degs['gene_id'].dropna().unique().tolist()

# Export list for DAVID, KEGG Mapper, and Cytoscape/BINGO manual upload
with open('deg_gene_list.txt', 'w') as f:
    for gene in gene_list:
        f.write(f"{gene}\n")
print("Gene list exported to 'deg_gene_list.txt' for DAVID, KEGG, and Cytoscape input.")

# Define the correct library name for C. elegans
go_library = 'GO_Biological_Process_2018' 

try:
    enr = gp.enrichr(gene_list=gene_list,
                     gene_sets=[go_library],
                     organism='celegans',
                     outdir=None)
    
    # Filter for FDR adjusted p-value < 0.05
    significant_go = enr.results[enr.results['Adjusted P-value'] < 0.05]
    print("\nTop Significant GO Terms:")
    display(significant_go[['Term', 'Overlap', 'Adjusted P-value']].head())
    
except Exception as e:
    print(f"Enrichr API call failed: {e}")
    print("\nTo see valid library names for C. elegans, run:")
    print("print(gp.get_library_name(organism='celegans'))")

Gene list exported to 'deg_gene_list.txt' for DAVID, KEGG, and Cytoscape input.

Top Significant GO Terms:


,Term,Overlap,Adjusted P-value


In [12]:
import gseapy as gp
from IPython.display import display

# 1. Try to use the 'gene' column (symbols) instead of 'gene_id' (Ensembl IDs)
if 'gene' in degs.columns:
    # Get unique gene symbols, drop NaN, and filter out placeholder '-' values Cuffdiff sometimes uses
    gene_list = degs[degs['gene'] != '-']['gene'].dropna().unique().tolist()
else:
    gene_list = degs['gene_id'].dropna().unique().tolist()

# 2. Print a sanity check
print(f"Number of genes submitted to Enrichr: {len(gene_list)}")
print(f"Sample of gene names: {gene_list[:5]}\n")

# 3. Export for manual web tools
with open('deg_gene_list.txt', 'w') as f:
    for gene in gene_list:
        f.write(f"{gene}\n")

# 4. Programmatic GO Enrichment
go_library = 'GO_Biological_Process_2018' 

try:
    enr = gp.enrichr(gene_list=gene_list,
                     gene_sets=[go_library],
                     organism='celegans',
                     outdir=None)
    
    # Check if we got ANY results back
    if enr.results.empty:
        print("Enrichr returned no results. The gene identifiers might not be recognized by the database.")
    else:
        # Filter for FDR adjusted p-value < 0.05
        significant_go = enr.results[enr.results['Adjusted P-value'] < 0.05]
        
        if significant_go.empty:
            print("No GO terms met the Adjusted P-value < 0.05 threshold.")
            print("Here are the top 5 terms BEFORE p-value filtering:")
            display(enr.results[['Term', 'Overlap', 'P-value', 'Adjusted P-value']].head())
        else:
            print(f"\nFound {len(significant_go)} Significant GO Terms (FDR < 0.05):")
            display(significant_go[['Term', 'Overlap', 'Adjusted P-value']].head())
            
except Exception as e:
    print(f"Enrichr API call failed: {e}")

Number of genes submitted to Enrichr: 4821
Sample of gene names: ['Y48G1C.10', 'mex-3', 'snpc-3.3', 'snpc-3.2', 'rnp-8']


Found 2 Significant GO Terms (FDR < 0.05):


,Term,Overlap,Adjusted P-value
0,sensory perception of chemical stimulus (GO:00...,87/231,0.001326
1,detection of chemical stimulus involved in sen...,87/217,0.000116
